# Spark DataFrame Analyse von Produktbewertungen

In diesem Notebook wird eine einfache Textanalyse mit Apache Spark DataFrames durchgeführt.

Ziel:
- Beispielbewertungen in ein Spark DataFrame laden
- Texte bereinigen
- Wörter extrahieren
- Wortfrequenzen mit Spark SQL berechnen

Die Verarbeitung erfolgt lokal mit PySpark.

### Python-Umgebung für PySpark konfigurieren
Optional kann die Python-Umgebung festgelegt werden. Die Umgebungsvariable `PYSPARK_PYTHON` zeigt auf die Python-Executable der gewünschten Umgebung (hier Conda).

In [ ]:
import os
import sys

python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

### Spark Session starten

Im folgenden wird eine `SparkSession` erstellt, die den Einstiegspunkt für alle Spark-Operationen wie DataFrames und SQL-Abfragen darstellt. Die Session läuft lokal (`local[*]`) und nutzt alle verfügbaren CPU-Kerne. 

In [ ]:
from pyspark.sql import SparkSession

# Spark-Session starten
spark = SparkSession.builder \
    .appName("ReviewsExampleDataFrame") \
    .master("local[*]") \
    .getOrCreate()

### Beispieldaten für Reviews erzeugen

In dieser Zelle werden Beispielbewertungen erstellt.  
Jeder Eintrag besteht aus:
- einer Bewertung (rating)
- einem Review-Text

Diese Daten dienen als Grundlage für die spätere Analyse und werden anschließend in ein Spark DataFrame umgewandelt.

In [ ]:
# Daten vorbereiten
data = [
    (5, "Der Laptop ist schnell und leise, bin sehr zufrieden."),
    (4, "Gute Qualität, aber Lieferung hat lange gedauert."),
    (1, "Enttäuschend – Gerät war defekt."),
    (5, "Top Produkt, gerne wieder!"),
    (3, "Okay, aber Preis etwas zu hoch.")
]

# DataFrame anlegen
df = spark.createDataFrame(data, ["rating", "review"])

# Anzeigen & Schema prüfen
df.show(truncate=False)
df.printSchema()

### Daten filtern und Text vorbereiten

Hier werden die Reviews verarbeitet:

1. Filtern der Bewertungen: nur Reviews mit Rating ≥ 4 werden berücksichtigt.
2. Textbereinigung: der Review-Text wird:
   - in Kleinbuchstaben umgewandelt
   - von Sonderzeichen bereinigt

Dadurch entsteht eine neue Spalte `clean_review`, die sich besser für Textanalysen eignet.

In [ ]:
from pyspark.sql.functions import col, split, explode, regexp_replace, lower

df_filtered = df.filter(col("rating") >= 4)

df_cleaned = df_filtered.withColumn(
    "clean_review",
    lower(regexp_replace(col("review"), r"[^a-zA-ZäöüÄÖÜß0-9\s]", ""))  # nur Buchstaben/Zahlen/Leerzeichen
)

# Reviews in Wörter zerlegen und zählen
df_words = (
    df_cleaned
    .withColumn("word", explode(split(col("clean_review"), r"\s+")))  # in Wörter aufspalten
    .filter(col("word") != "")  # leere Strings entfernen
    .groupBy("word")
    .count()
    .orderBy(col("count").desc())
)
print("Wortzählung aus gefilterten Reviews (bereinigt):")
df_words.show(truncate=False)

### Wortfrequenzen mit Spark SQL berechnen

Das DataFrame wird als temporäre SQL-View registriert, sodass SQL-Abfragen darauf ausgeführt werden können.

Die SQL-Abfrage:
1. bereinigt den Text
2. zerlegt ihn mit `split()` in einzelne Wörter
3. nutzt `explode()`, um jedes Wort als eigene Zeile darzustellen
4. zählt anschließend die Häufigkeit jedes Wortes

Das Ergebnis ist eine einfache Wortfrequenzanalyse der Reviews.

In [ ]:
df.createOrReplaceTempView("df")

spark.sql("""SELECT
  word,
  COUNT(*) AS count
FROM (
  SELECT explode(
           split(
             lower(regexp_replace(coalesce(review, ''), '[^a-zA-ZäöüÄÖÜß0-9\\\\s]', '')),
             '\\\\s+'
           )
         ) AS word
  FROM df
  WHERE rating >= 4
) t
WHERE word <> ''
GROUP BY word
ORDER BY count DESC""").show()

### Aufräumen der Ressourcen

Zum Abschluss werden die verwendeten Spark-Objekte entfernt und die SparkSession beendet.  

In [ ]:
# DataFrame löschen
del df

# temporäre SQL View entfernen
spark.catalog.dropTempView("df")

# Spark Session stoppen
spark.stop()

print("Spark Session beendet und Daten bereinigt.")